## 01_load_and_prepare

In [ ]:
import pandas as pd
import numpy as np

import os

DATA_PATH = "data"

files = [f for f in os.listdir(DATA_PATH) if f.endswith(".csv")]

selected_file = st.sidebar.selectbox(
    "Selecciona dataset",
    files
)

df = pd.read_csv(os.path.join(DATA_PATH, selected_file))



# Columnas dinámicas
req_cols = [c for c in df.columns if c.startswith("req_")]
coverage_cols = [c for c in df.columns if c.startswith("stcov")]

# Métricas agregadas
df["mean_coverage"] = df[coverage_cols].mean(axis=1)
df["min_coverage"] = df[coverage_cols].min(axis=1)
df["max_coverage"] = df[coverage_cols].max(axis=1)

print(df.head())

   effort  satisfaction  req_1  req_2  req_3  req_4  req_5  req_6  req_7  \
0       1          62.0      1      0      0      0      0      0      0   
1       2         118.0      1      0      0      0      0      0      0   
2       3         174.0      1      0      0      0      0      0      0   
3       5         224.0      1      0      0      0      0      0      0   
4       6         228.0      1      0      0      0      0      0      0   

   req_8  ...  productivity  squandering  stcov_c1  stcov_c2  stcov_c3  \
0      0  ...          62.0     0.988372  0.064516  0.065574  0.078125   
1      0  ...          59.0     0.976744  0.129032  0.131148  0.140625   
2      1  ...          58.0     0.965116  0.193548  0.196721  0.203125   
3      1  ...          44.8     0.941860  0.225806  0.262295  0.281250   
4      1  ...          38.0     0.930233  0.258065  0.262295  0.265625   

   stcov_c4  stcov_c5  mean_coverage  min_coverage  max_coverage  
0  0.061538  0.075758       0.0

## 02_pareto_front

In [ ]:
import pandas as pd
import plotly.express as px

# df = pd.read_csv("dataset.csv")

coverage_cols = [c for c in df.columns if c.startswith("stcov")]

df["mean_coverage"] = df[coverage_cols].mean(axis=1)

# --------------------------------------------
# Pareto detection
# --------------------------------------------

def pareto_front(df):

    pareto = []

    for i, row in df.iterrows():

        dominated = False

        for j, other in df.iterrows():

            if i == j:
                continue

            better_or_equal = (
                other["satisfaction"] >= row["satisfaction"]
                and other["effort"] <= row["effort"]
                and other["mean_coverage"] >= row["mean_coverage"]
            )

            strictly_better = (
                other["satisfaction"] > row["satisfaction"]
                or other["effort"] < row["effort"]
                or other["mean_coverage"] > row["mean_coverage"]
            )

            if better_or_equal and strictly_better:
                dominated = True
                break

        pareto.append(not dominated)

    return pareto

df["pareto"] = pareto_front(df)

# --------------------------------------------
# Scatter Pareto
# --------------------------------------------

fig = px.scatter(
    df,
    x="effort",
    y="satisfaction",
    color="pareto",
    size="mean_coverage",
    hover_data=[
        "id",
        "productivity",
        "squandering",
        "mean_coverage"
    ],
    title="Pareto Front Analysis",
)

fig.update_layout(
    template="plotly_white",
    height=700
)

fig.show()

## 03_customer_coverage

In [3]:
import pandas as pd
import plotly.graph_objects as go

# df = pd.read_csv("dataset.csv")

coverage_cols = [c for c in df.columns if c.startswith("stcov")]

fig = go.Figure()

for col in coverage_cols:

    fig.add_trace(
        go.Scatter(
            x=df["id"],
            y=df[col],
            mode="lines+markers",
            name=col
        )
    )

fig.update_layout(
    title="Customer Coverage Evolution",
    xaxis_title="Solution ID",
    yaxis_title="Coverage",
    template="plotly_white",
    height=700
)

fig.show()

## 04_requirements_heatmap

In [4]:
import pandas as pd
import plotly.express as px

# df = pd.read_csv("dataset.csv")

req_cols = [c for c in df.columns if c.startswith("req_")]

fig = px.imshow(
    df[req_cols],
    aspect="auto",
    color_continuous_scale="Viridis",
    labels=dict(
        x="Requirements",
        y="Solutions",
        color="Activated"
    ),
    title="Requirements Activation Heatmap"
)

fig.update_layout(
    template="plotly_white",
    height=800
)

fig.show()

## 05_parallel_coordinates

In [5]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("dataset.csv")

coverage_cols = [c for c in df.columns if c.startswith("stcov")]

df["mean_coverage"] = df[coverage_cols].mean(axis=1)

fig = px.parallel_coordinates(
    df,
    dimensions=[
        "satisfaction",
        "effort",
        "productivity",
        "squandering",
        "mean_coverage"
    ],
    color="satisfaction",
)

fig.update_layout(
    title="Multiobjective Parallel Coordinates"
)

fig.show()

## 06_radar_chart

In [6]:
import pandas as pd
import plotly.graph_objects as go

# df = pd.read_csv("dataset.csv")

coverage_cols = [c for c in df.columns if c.startswith("stcov")]

df["mean_coverage"] = df[coverage_cols].mean(axis=1)

# Seleccionar soluciones
selected = [0, len(df)//2, len(df)-1]

fig = go.Figure()

for idx in selected:

    row = df.iloc[idx]

    values = [
        row["satisfaction"],
        row["productivity"],
        row["mean_coverage"],
        1 - row["squandering"],
        1 / row["effort"]
    ]

    categories = [
        "Satisfaction",
        "Productivity",
        "Coverage",
        "Efficiency",
        "Low Effort"
    ]

    fig.add_trace(
        go.Scatterpolar(
            r=values,
            theta=categories,
            fill='toself',
            name=f'Solution {row["id"]}'
        )
    )

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    title="Solution Comparison Radar",
    template="plotly_white"
)

fig.show()

## 07_export_html

In [7]:
# Guardar cualquier figura plotly a HTML

fig.write_html(
    "pareto_dashboard.html",
    include_plotlyjs="cdn"
)